In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

import warnings
warnings.simplefilter('ignore')

import os
import sys
import subprocess

def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

# ============================================================================
# Tree of Thoughts Components (Enhanced)
# ============================================================================

class ThoughtNode:
    """Represents a node in the thought tree"""
    
    def __init__(self, content: str, parent=None, depth: int = 0):
        self.content = content
        self.parent = parent
        self.children = []
        self.depth = depth
        self.value_score = 0.0
        self.entropy_score = float('inf')
        self.answer = None
        self.python_calls = 0
        self.python_errors = 0
        self.is_terminal = False
        self.is_pruned = False
        self.exploration_count = 0
        self.quality_metrics = {
            'reasoning_coherence': 0.0,
            'mathematical_rigor': 0.0,
            'progress_score': 0.0
        }
        
    def add_child(self, child_node):
        self.children.append(child_node)
        
    def get_path(self) -> list:
        """Get the path from root to this node"""
        path = []
        current = self
        while current is not None:
            path.append(current)
            current = current.parent
        return list(reversed(path))
    
    def get_path_text(self) -> str:
        """Get concatenated text of reasoning path"""
        path = self.get_path()
        return "\n\n".join([f"Step {i}: {node.content}" for i, node in enumerate(path[1:])])
    
    def compute_combined_score(self) -> float:
        """Compute combined score for node selection"""
        base_score = 1.0 / max(self.entropy_score, 1e-9)
        
        # Bonus for having valid answer
        if self.answer is not None:
            base_score *= 5.0
            
        # Penalty for errors
        if self.python_errors > 0:
            base_score *= (0.5 ** self.python_errors)
        
        # Bonus for successful python usage
        if self.python_calls > 0 and self.python_errors == 0:
            base_score *= 1.5
            
        # Quality metrics bonus
        quality_avg = sum(self.quality_metrics.values()) / max(len(self.quality_metrics), 1)
        base_score *= (1.0 + quality_avg)
        
        # Depth penalty (prefer deeper, more developed thoughts)
        base_score *= (1.0 + 0.1 * self.depth)
        
        return base_score
    
    def compute_ucb_score(self, total_explorations: int, exploration_constant: float = 1.414) -> float:
        """Compute Upper Confidence Bound score for node selection"""
        if self.exploration_count == 0:
            return float('inf')
        
        exploitation = self.value_score
        exploration = exploration_constant * math.sqrt(
            math.log(total_explorations) / self.exploration_count
        )
        
        return exploitation + exploration

class TreeOfThoughts:
    """Pure Tree of Thoughts reasoning framework"""
    
    def __init__(
        self, 
        max_depth: int = 4, 
        branching_factor: int = 3,
        beam_width: int = 5,
        pruning_threshold: float = 0.1
    ):
        self.max_depth = max_depth
        self.branching_factor = branching_factor
        self.beam_width = beam_width
        self.pruning_threshold = pruning_threshold
        self.root = None
        self.all_nodes = []
        self.total_explorations = 0
        
    def initialize(self, initial_thought: str):
        """Initialize the tree with root node"""
        self.root = ThoughtNode(initial_thought, parent=None, depth=0)
        self.all_nodes = [self.root]
        
    def expand_node(self, node: ThoughtNode, expansion_prompts: list) -> list:
        """Expand a node by generating child thoughts"""
        children = []
        
        for prompt in expansion_prompts[:self.branching_factor]:
            child = ThoughtNode(
                content=prompt,
                parent=node,
                depth=node.depth + 1
            )
            node.add_child(child)
            children.append(child)
            self.all_nodes.append(child)
            
        return children
    
    def get_best_leaf_nodes(self, k: int = 5) -> list:
        """Get top-k leaf nodes based on scores"""
        leaf_nodes = [n for n in self.all_nodes 
                     if (not n.children or n.is_terminal) and not n.is_pruned]
        
        scored_nodes = [(n, n.compute_combined_score()) for n in leaf_nodes]
        scored_nodes.sort(key=lambda x: x[1], reverse=True)
        
        return [n for n, _ in scored_nodes[:k]]
    
    def get_nodes_at_depth(self, depth: int) -> list:
        """Get all non-pruned nodes at a specific depth"""
        return [n for n in self.all_nodes 
                if n.depth == depth and not n.is_pruned]
    
    def get_best_nodes_for_expansion(self, k: int = 3) -> list:
        """Get best nodes for further expansion using beam search"""
        expandable_nodes = [
            n for n in self.all_nodes 
            if not n.is_terminal 
            and not n.is_pruned 
            and n.depth < self.max_depth
            and len(n.children) < self.branching_factor
        ]
        
        if not expandable_nodes:
            return []
        
        # Use UCB for exploration-exploitation balance
        scored_nodes = [
            (n, n.compute_ucb_score(self.total_explorations)) 
            for n in expandable_nodes
        ]
        scored_nodes.sort(key=lambda x: x[1], reverse=True)
        
        return [n for n, _ in scored_nodes[:k]]
    
    def prune_low_quality_nodes(self):
        """Prune nodes with low quality scores"""
        if len(self.all_nodes) <= self.beam_width:
            return
        
        # Get all non-terminal nodes
        non_terminal = [n for n in self.all_nodes if not n.is_terminal]
        
        if not non_terminal:
            return
        
        scores = [n.compute_combined_score() for n in non_terminal]
        threshold = sorted(scores, reverse=True)[min(self.beam_width, len(scores)-1)]
        threshold *= self.pruning_threshold
        
        for node in non_terminal:
            if node.compute_combined_score() < threshold and node != self.root:
                node.is_pruned = True
    
    def get_best_path(self) -> list:
        """Get the single best reasoning path"""
        best_leaves = self.get_best_leaf_nodes(k=1)
        
        if not best_leaves:
            return [self.root]
        
        return best_leaves[0].get_path()
    
    def get_all_answers(self) -> list:
        """Get all valid answers from the tree"""
        answers = []
        for node in self.all_nodes:
            if node.answer is not None and not node.is_pruned:
                score = node.compute_combined_score()
                answers.append((node.answer, score, node))
        
        answers.sort(key=lambda x: x[1], reverse=True)
        return answers

class ToTPromptGenerator:
    """Generate prompts for Tree of Thoughts exploration"""
    
    @staticmethod
    def generate_initial_prompts(problem: str) -> list:
        """Generate initial diverse approaches to the problem"""
        return [
            f"Approach 1 - Direct Analysis:\nAnalyze this problem step by step, identifying key mathematical structures:\n{problem}",
            f"Approach 2 - Pattern Recognition:\nLook for patterns, symmetries, or special cases in:\n{problem}",
            f"Approach 3 - Decomposition:\nBreak down this problem into smaller sub-problems:\n{problem}",
        ]
    
    @staticmethod
    def generate_expansion_prompts(node: ThoughtNode, problem: str) -> list:
        """Generate prompts for expanding a thought node"""
        path_text = node.get_path_text()
        
        return [
            f"Continue this reasoning path:\n{path_text}\n\nNext step for: {problem}",
            f"Verify and refine this approach:\n{path_text}\n\nFor: {problem}",
            f"Explore alternative direction from:\n{path_text}\n\nProblem: {problem}",
        ]
    
    @staticmethod
    def generate_refinement_prompt(node: ThoughtNode, problem: str) -> str:
        """Generate prompt to refine a specific reasoning path"""
        path_text = node.get_path_text()
        return (
            f"Refine and complete this solution approach:\n\n{path_text}\n\n"
            f"Original problem: {problem}\n\n"
            f"Provide detailed calculations and the final answer in \\boxed{{}} format."
        )
    
    @staticmethod
    def generate_synthesis_prompt(nodes: list, problem: str) -> str:
        """Generate prompt to synthesize multiple reasoning paths"""
        paths_text = "\n\n".join([
            f"Path {i+1} (Score: {node.compute_combined_score():.2f}):\n{node.get_path_text()}" 
            for i, node in enumerate(nodes)
        ])
        
        return (
            f"Synthesize these reasoning approaches to find the final answer:\n\n"
            f"Problem: {problem}\n\n"
            f"Available reasoning paths:\n{paths_text}\n\n"
            f"Combine the best insights from each path and provide the final answer in \\boxed{{}} format."
        )

# ============================================================================
# Configuration with Pure ToT parameters
# ============================================================================

class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 100000
    buffer_tokens = 1000
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.8  # Slightly lower for more focused reasoning
    min_p = 0.02
    
    # Pure Tree of Thoughts parameters
    tot_max_depth = 4
    tot_branching_factor = 3
    tot_beam_width = 8
    tot_pruning_threshold = 0.15
    tot_expansion_rounds = 6
    tot_refinement_attempts = 3
    tot_synthesis_enabled = True

set_seed(CFG.seed)

# ============================================================================
# Original classes (unchanged)
# ============================================================================

class AIMO3Template:

    def __init__(self):
        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

# ============================================================================
# Pure Tree of Thoughts Solver
# ============================================================================

class PureToTSolver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        
        # Initialize ToT components
        self.tot_prompt_gen = ToTPromptGenerator()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _evaluate_node_quality(self, node: ThoughtNode, response_text: str):
        """Evaluate the quality of reasoning in a node"""
        
        # Simple heuristics for quality assessment
        text_lower = response_text.lower()
        
        # Reasoning coherence: presence of logical connectors
        coherence_indicators = ['therefore', 'thus', 'hence', 'because', 'since', 'so']
        coherence_score = sum(1 for ind in coherence_indicators if ind in text_lower) / len(coherence_indicators)
        
        # Mathematical rigor: presence of mathematical language
        rigor_indicators = ['equation', 'formula', 'theorem', 'proof', 'let', 'assume', 'given']
        rigor_score = sum(1 for ind in rigor_indicators if ind in text_lower) / len(rigor_indicators)
        
        # Progress: length and structure
        progress_score = min(len(response_text) / 1000, 1.0)
        
        node.quality_metrics['reasoning_coherence'] = coherence_score
        node.quality_metrics['mathematical_rigor'] = rigor_score
        node.quality_metrics['progress_score'] = progress_score
    
    def _process_tot_node(
        self,
        node: ThoughtNode,
        problem: str,
        system_prompt: str,
        stop_event: threading.Event,
        deadline: float,
        temperature: float = None
    ) -> ThoughtNode:
        """Process a single ToT node and populate its metrics"""
        
        if stop_event.is_set() or time.time() > deadline:
            return node
        
        if temperature is None:
            temperature = self.cfg.temperature
        
        sandbox = None
        local_tool = None
        logprobs_buffer = []
        full_response_text = []
        
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox
            )
            
            # Construct prompt from node path
            path = node.get_path()
            thought_context = "\n\n".join([f"Step {i}: {n.content}" for i, n in enumerate(path[1:])])
            
            full_prompt = f"{problem}\n\n{self.cfg.preference_prompt}\n\nCurrent reasoning path:\n{thought_context}"
            
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt,
                full_prompt,
                local_tool.tool_config
            )
            
            conversation = Conversation.from_messages(messages)
            
            for turn_idx in range(self.cfg.turns // 2):
                if stop_event.is_set() or time.time() > deadline:
                    break
                
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                
                if max_tokens < self.cfg.buffer_tokens:
                    break
                
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=self.cfg.seed + node.depth * 1000 + turn_idx,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True
                    }
                )
                
                try:
                    token_buffer = []
                    text_chunks = []
                    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
                        
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
                        
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            text_chunks.append(new_text)
                            full_response_text.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
                        
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            
                            if answer is not None:
                                node.answer = answer
                                node.is_terminal = True
                                break
                
                finally:
                    stream.close()
                
                if node.is_terminal:
                    break
                
                if not token_buffer:
                    break
                
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
                
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    node.answer = self._scan_for_answer(answer_text)
                    node.is_terminal = True
                    break
                
                if last_message.recipient == 'python':
                    node.python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    
                    response_text = tool_responses[0].content[0].text
                    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text:
                        node.python_errors += 1
                    
                    conversation.messages.extend(tool_responses)
        
        except Exception as exc:
            node.python_errors += 1
            print(f"Node processing error: {exc}")
        
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
        
        node.entropy_score = self._compute_mean_entropy(logprobs_buffer)
        node.exploration_count += 1
        
        # Evaluate quality
        response_full = ''.join(full_response_text)
        self._evaluate_node_quality(node, response_full)
        
        # Update value score
        node.value_score = node.compute_combined_score()
        
        return node
    
    def _pure_tot_search(
        self,
        problem: str,
        system_prompt: str,
        deadline: float,
        stop_event: threading.Event
    ) -> TreeOfThoughts:
        """Pure Tree of Thoughts search as primary method"""
        
        print("=" * 80)
        print("PURE TREE OF THOUGHTS SEARCH")
        print("=" * 80)
        
        # Initialize tree
        tot = TreeOfThoughts(
            max_depth=self.cfg.tot_max_depth,
            branching_factor=self.cfg.tot_branching_factor,
            beam_width=self.cfg.tot_beam_width,
            pruning_threshold=self.cfg.tot_pruning_threshold
        )
        
        tot.initialize(f"Problem: {problem}")
        
        # Phase 1: Initial diverse exploration
        print("\n[Phase 1] Initial Exploration - Generating diverse approaches...")
        initial_prompts = self.tot_prompt_gen.generate_initial_prompts(problem)
        initial_children = tot.expand_node(tot.root, initial_prompts)
        
        # Process initial nodes
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        futures = []
        
        for node in initial_children:
            future = executor.submit(
                self._process_tot_node,
                node,
                problem,
                system_prompt,
                stop_event,
                deadline,
                self.cfg.temperature
            )
            futures.append((future, node))
        
        for future, node in futures:
            try:
                future.result()
                tot.total_explorations += 1
            except Exception as exc:
                print(f"Initial node failed: {exc}")
        
        executor.shutdown(wait=True)
        
        print(f"Initial exploration complete. Found {len([n for n in initial_children if n.answer])} answers.")
        
        # Phase 2: Iterative expansion and refinement
        for round_idx in range(self.cfg.tot_expansion_rounds):
            if stop_event.is_set() or time.time() > deadline:
                break
            
            print(f"\n[Phase 2.{round_idx + 1}] Expansion Round {round_idx + 1}/{self.cfg.tot_expansion_rounds}")
            
            # Select best nodes for expansion
            nodes_to_expand = tot.get_best_nodes_for_expansion(k=self.cfg.tot_beam_width)
            
            if not nodes_to_expand:
                print("No more nodes to expand.")
                break
            
            print(f"Expanding {len(nodes_to_expand)} promising nodes...")
            
            # Expand selected nodes
            new_nodes = []
            for node in nodes_to_expand:
                expansion_prompts = self.tot_prompt_gen.generate_expansion_prompts(node, problem)
                children = tot.expand_node(node, expansion_prompts)
                new_nodes.extend(children)
            
            # Process new nodes in parallel
            executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
            futures = []
            
            for node in new_nodes:
                future = executor.submit(
                    self._process_tot_node,
                    node,
                    problem,
                    system_prompt,
                    stop_event,
                    deadline,
                    self.cfg.temperature * (0.9 ** round_idx)  # Decrease temperature over rounds
                )
                futures.append((future, node))
            
            answers_found = 0
            for future, node in futures:
                try:
                    processed = future.result()
                    tot.total_explorations += 1
                    if processed.answer is not None:
                        answers_found += 1
                except Exception as exc:
                    print(f"Node expansion failed: {exc}")
            
            executor.shutdown(wait=True)
            
            print(f"Round {round_idx + 1} complete. Found {answers_found} new answers.")
            
            # Prune low-quality branches
            tot.prune_low_quality_nodes()
            active_nodes = len([n for n in tot.all_nodes if not n.is_pruned])
            print(f"Active nodes after pruning: {active_nodes}")
            
            # Check if we have strong consensus
            all_answers = tot.get_all_answers()
            if all_answers:
                top_answers = [a for a, _, _ in all_answers[:3]]
                if len(set(top_answers)) == 1 and len(all_answers) >= 3:
                    print(f"Strong consensus reached on answer: {top_answers[0]}")
                    break
        
        # Phase 3: Refinement of best paths
        print("\n[Phase 3] Refinement - Deepening best reasoning paths...")
        best_nodes = tot.get_best_leaf_nodes(k=self.cfg.tot_refinement_attempts)
        
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        futures = []
        
        for node in best_nodes:
            refinement_prompt = self.tot_prompt_gen.generate_refinement_prompt(node, problem)
            
            # Create refinement child
            refinement_node = ThoughtNode(
                content=refinement_prompt,
                parent=node,
                depth=node.depth + 1
            )
            node.add_child(refinement_node)
            tot.all_nodes.append(refinement_node)
            
            future = executor.submit(
                self._process_tot_node,
                refinement_node,
                problem,
                system_prompt,
                stop_event,
                deadline,
                self.cfg.temperature * 0.7  # Lower temperature for refinement
            )
            futures.append((future, refinement_node))
        
        for future, node in futures:
            try:
                future.result()
                tot.total_explorations += 1
            except Exception as exc:
                print(f"Refinement failed: {exc}")
        
        executor.shutdown(wait=True)
        
        # Phase 4: Synthesis (if enabled and no clear answer)
        if self.cfg.tot_synthesis_enabled:
            all_answers = tot.get_all_answers()
            
            if not all_answers or len(set([a for a, _, _ in all_answers])) > 2:
                print("\n[Phase 4] Synthesis - Combining insights from multiple paths...")
                
                best_paths = tot.get_best_leaf_nodes(k=3)
                synthesis_prompt = self.tot_prompt_gen.generate_synthesis_prompt(best_paths, problem)
                
                synthesis_node = ThoughtNode(
                    content=synthesis_prompt,
                    parent=tot.root,
                    depth=1
                )
                tot.root.add_child(synthesis_node)
                tot.all_nodes.append(synthesis_node)
                
                self._process_tot_node(
                    synthesis_node,
                    problem,
                    system_prompt,
                    stop_event,
                    deadline,
                    self.cfg.temperature * 0.5
                )
        
        print("\n" + "=" * 80)
        print("TREE SEARCH COMPLETE")
        print("=" * 80)
        
        return tot
    
    def _select_final_answer(self, tot: TreeOfThoughts) -> int:
        """Select final answer from tree of thoughts"""
        
        all_answers = tot.get_all_answers()
        
        if not all_answers:
            print("\nNo valid answers found in tree. Returning 0.")
            return 0
        
        # Display answer statistics
        print(f"\nFound {len(all_answers)} answer candidates:")
        
        answer_summary = []
        for answer, score, node in all_answers[:10]:
            answer_summary.append({
                'Answer': answer,
                'Score': round(score, 3),
                'Depth': node.depth,
                'Entropy': round(node.entropy_score, 3),
                'Python Calls': node.python_calls,
                'Errors': node.python_errors
            })
        
        summary_df = pd.DataFrame(answer_summary)
        display(summary_df)
        
        # Two-stage voting: frequency is the primary signal, average node score
        # (normalized per answer) is the tiebreaker.  This prevents a single
        # high-scoring outlier node from outranking an answer that appears
        # consistently across many nodes.
        answer_counts = Counter(answer for answer, _, _ in all_answers)
        answer_score_sum = defaultdict(float)

        for answer, score, node in all_answers:
            answer_score_sum[answer] += score

        # Combined weight = count * (average score for that answer)
        # Using average rather than sum means a lone high-score node cannot
        # beat a frequent low-entropy answer simply by accumulating raw score.
        answer_weights = {
            ans: count * (answer_score_sum[ans] / count)
            for ans, count in answer_counts.items()
        }

        # Primary sort: frequency (count); secondary sort: average score
        sorted_answers = sorted(
            answer_weights.items(),
            key=lambda x: (answer_counts[x[0]], x[1]),
            reverse=True
        )

        vote_df = pd.DataFrame(
            [
                (ans, answer_counts[ans], round(answer_score_sum[ans] / answer_counts[ans], 3))
                for ans, _ in sorted_answers[:5]
            ],
            columns=['Answer', 'Count', 'Avg Score']
        )

        print("\nFrequency-first voting results:")
        display(vote_df)
        
        final_answer = sorted_answers[0][0]
        print(f"\n{'='*80}")
        print(f"FINAL ANSWER: {final_answer}")
        print(f"{'='*80}\n")
        
        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\n{"="*80}')
        print(f'PROBLEM: {problem}')
        print(f'{"="*80}\n')
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Time Budget: {budget:.2f} seconds')
        print(f'Deadline: {time.strftime("%H:%M:%S", time.localtime(deadline))}\n')
    
        stop_event = threading.Event()
        
        gc.disable()
        
        try:
            # Pure Tree of Thoughts search
            tot = self._pure_tot_search(
                problem,
                self.cfg.system_prompt,
                deadline,
                stop_event
            )
            
            # Select final answer
            final_answer = self._select_final_answer(tot)
            
        finally:
            stop_event.set()
            self.problems_remaining = max(0, self.problems_remaining - 1)
            gc.enable()
            gc.collect()
    
        return final_answer
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

solver = PureToTSolver(CFG)

def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
    )